In [1]:
import pandas as pd
import joblib
import os
import sys
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score

sys.path.append("..")
from src.preprocessing import clean_text

# --- Load and prepare data (same as Week 1-2) ---
df = pd.read_csv("../data/Consumer_Reviews_of_Amazon_Products.csv")
df = df.dropna(subset=['reviews.text', 'reviews.rating'])
df = df.drop_duplicates(subset=['reviews.text'])

def rating_to_sentiment(rating):
    if rating <= 2:
        return "Negative"
    elif rating == 3:
        return "Neutral"
    else:
        return "Positive"

df['sentiment'] = df['reviews.rating'].apply(rating_to_sentiment)

X = df['reviews.text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_clean = X_train.apply(clean_text)
X_test_clean = X_test.apply(clean_text)

# --- TF-IDF (same settings as Week 2) ---
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train_clean)
X_test_tfidf = vectorizer.transform(X_test_clean)

# --- Train final model: calibrated Linear SVM (Week 3 decision) ---
final_model = CalibratedClassifierCV(LinearSVC(class_weight='balanced', random_state=42))
final_model.fit(X_train_tfidf, y_train)

# Sanity check — should roughly match your Week 3 SVM macro-F1 (0.53)
y_pred_final = final_model.predict(X_test_tfidf)
print(f"Final model macro-F1: {f1_score(y_test, y_pred_final, average='macro'):.3f}")

# --- Save vectorizer and model for the app ---
os.makedirs("../models", exist_ok=True)
joblib.dump(vectorizer, "../models/vectorizer.pkl")
joblib.dump(final_model, "../models/classifier.pkl")
print("Saved vectorizer.pkl and classifier.pkl to models/")


import numpy as np

coefs = np.mean([c.estimator.coef_ for c in final_model.calibrated_classifiers_], axis=0)
feature_names = np.array(vectorizer.get_feature_names_out())

for i, label in enumerate(final_model.classes_):
    top = np.argsort(coefs[i])[-10:][::-1]
    print(label, "->", list(feature_names[top]))

C:\Users\Admin\AppData\Local\Temp\ipykernel_7556\4189257234.py:15: DtypeWarning: Columns (1: name, 10: reviews.didPurchase) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/Consumer_Reviews_of_Amazon_Products.csv")


Final model macro-F1: 0.424
Saved vectorizer.pkl and classifier.pkl to models/
Negative -> ['not worth', 'price awesome', 'returning', 'returned', 'match', 'apart', 'later', 'felt', 'constantly', 'doesnt']
Neutral -> ['kindle light', 'wouldn recommend', 'just not', 'rate', 'instagram', 'include', 'play store', 'initial', 'tied', 'premium']
Positive -> ['love', 'great', 'perfect', 'awesome', 'easy', 'not disappointed', 'excellent', 'pleased', 'especially', 'amazing']
